# 04. Credit Default Release, Serving, and Monitoring

Notebooks 01–03 covered **Stages 1–3 (`BUILD & VALIDATE`)** in `DEV`: exploring source contracts, registering pinned Feature Views (`V01`) and immutable Datasets, and running tracked experiments to select the frozen `HistGradientBoostingClassifier` candidate.

This notebook showcases **Stages 4–7 (`SERVE & OPERATE`)** using the shared `credit_default` package (`src/credit_default/`), the Code Bundle entrypoints (`jobs/train.py`, `jobs/deploy_service.py`), and the `CONTROL` governance tables and owner's-rights stored procedures (`ACTIVATE_SERVING`, `RECOVER_SERVING`):

1. **Loop 1 & Loop 2 — Code Bundles & Unattended Candidate Creation:** Validate the frozen pipeline release in `Pre-Prod` (`TEST`) and run the approved Code Bundle in `PROD` (`PROD_FEATURE_STORE`) to register a production candidate in `CONTROL.PIPELINE_ATTEMPT`.
2. **Loop 3 — Deterministic Screening vs. Accountable Owner Approval:** Prove that `screening_status = 'accepted'` does not change live serving and that `CONTROL.ACTIVATE_SERVING` fails closed without a matching `'approve'` record in `CONTROL.PROMOTION_DECISION`.
3. **Loop 4 (Recommended Default) — Warehouse Batch Inference:** Move the `PRODUCTION` model alias via `CONTROL.ACTIVATE_SERVING(..., 'BATCH', 100)` and score `RAW.SCORING_POPULATION` into `CONTROL.PREDICTION_RECORD`.
4. **Loop 4 (Online Reference Branch) — Stable Gateway (`SHADOW` vs. `SPLIT`):** Deploy a disposable SPCS model service with `autocapture=True` (`jobs/deploy_service.py`) and compare **Shadow Model** (`type: shadow_traffic`, Preview) against **Traffic Split** (`type: traffic_split`, GA) routing behind a permanent Gateway URL.
5. **Stages 6 & 7 — Monitoring, Delayed 90-Day Ground Truth & Full-Path Rollback:** Replay time across the `2026-01-01` drift boundary without treating `PENDING` labels as negative (`0`), record a governed operational action in `CONTROL.OPERATIONAL_ACTION`, and verify full-path rollback via `CONTROL.RECOVER_SERVING`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowflake.snowpark.context import get_active_session

project_root = Path("..").resolve()
if not (project_root / "project.yaml").is_file():
    project_root = Path(".").resolve()
sys.path.insert(0, str(project_root / "src"))

from credit_default import (
    activate_serving,
    evaluate_delayed_outcomes,
    load_project,
    load_workflow_contract,
    pipeline_release_decision,
    record_environment_deployment,
    record_operational_action,
    record_promotion_decision,
    record_source_release,
    recover_serving,
    resolve_binding,
    run_warehouse_batch_scoring,
    train_and_register_candidate,
)

session = get_active_session()
project = load_project(project_root / "project.yaml")
contract = load_workflow_contract(project)

database_name = project["snowflake"]["database"]
control_schema = project["snowflake"]["schemas"]["control"]
engineer_role = project["snowflake"]["roles"]["engineer"]
prod_owner_role = project["snowflake"]["roles"]["prod_owner"]
service_role = project["snowflake"]["roles"]["service"]

session.use_role(engineer_role)
session.use_warehouse(project["snowflake"]["warehouse"])
session.use_database(database_name)
session.use_schema(control_schema)

bindings_df = pd.DataFrame([resolve_binding(env, project) for env in ["DEV", "TEST", "PROD"]])
display(
    bindings_df[
        [
            "environment",
            "presentation_environment",
            "schema",
            "feature_store_schema",
            "model_schema",
            "may_approve_future_prod_model",
            "may_change_live_serving",
        ]
    ]
)

## Section 1 — Loop 1 & Loop 2: Code Bundles & Unattended Candidate Creation

A reviewed commit produces one immutable source release (`CONTROL.SOURCE_RELEASE`) that is deployed into environment-specific Code Bundles (`CREATE CODE BUNDLE`) across `TEST` (`Pre-Prod`) and `PROD` (`CONTROL.ENVIRONMENT_DEPLOYMENT`):

- **Loop 1 (Pipeline release — Gate 1 in `Pre-Prod`):** GitHub Actions (`snow dcm deploy` + `snow stage copy` + `snow sql`) executes `jobs/train.py` in `TEST` against `TEST_FEATURE_STORE`. Passing `TEST` approves the **code release** (`approve_pipeline`)—it never approves a future `PROD` model.
- **Loop 2 (Prod candidate — inside `PROD`):** A Snowflake Task Graph (`CRISK_DEMO_TRAIN_CANDIDATE_TASK`) executes the approved Code Bundle (`EXECUTE CODE BUNDLE ... ENTRYPOINT = 'jobs/train.py'`) against `PROD_FEATURE_STORE`, registers the candidate in `PROD`, and writes the screening outcome (`accepted`, `rejected`, or `failed`) to `CONTROL.PIPELINE_ATTEMPT`.

In [ ]:
release_id = "R_2026_10_01"
dataset_version = "V01"

session.use_role(engineer_role)
record_source_release(
    session,
    release_id=release_id,
    source_commit="main@2026-10-01",
    payload_digest="SHA256_ALLOWLISTED_BUNDLE_V01",
    project=project,
)
for env_name in ["TEST", "PROD"]:
    record_environment_deployment(
        session,
        release_id=release_id,
        environment_name=env_name,
        bundle_name=f"CRISK_DEMO_TRAIN_{release_id}",
        pipeline_validation_status="PENDING",
        project=project,
    )

session.use_role(service_role)

# Gate 1: Pre-Prod (TEST) pipeline-release validation
preprod_attempt = train_and_register_candidate(
    session,
    environment_name="TEST",
    release_id=release_id,
    dataset_version=dataset_version,
    source_state="main@2026-10-01",
    bundle_name=f"CRISK_DEMO_TRAIN_{release_id}",
)
preprod_gate = pipeline_release_decision(
    environment_name="TEST",
    checks_passed=(preprod_attempt["screening_status"] == "accepted"),
    project=project,
)

session.use_role(engineer_role)
session.sql(
    f"""
    UPDATE {database_name}.{control_schema}.ENVIRONMENT_DEPLOYMENT
    SET PIPELINE_VALIDATION_STATUS = ?
    WHERE RELEASE_ID = ? AND ENVIRONMENT = 'TEST'
    """,
    params=[preprod_gate["decision"], release_id],
).collect()

# Loop 2: Prod candidate creation (same code bundle, PROD_FEATURE_STORE bindings)
session.use_role(service_role)
prod_candidate = train_and_register_candidate(
    session,
    environment_name="PROD",
    release_id=release_id,
    dataset_version=dataset_version,
    source_state="main@2026-10-01",
    bundle_name=f"CRISK_DEMO_TRAIN_{release_id}",
)

attempts_df = session.sql(
    f"""
    SELECT
        ENVIRONMENT,
        RELEASE_ID,
        BUNDLE_NAME,
        FEATURE_CONTRACT,
        MODEL_SCHEMA,
        MODEL_NAME,
        MODEL_VERSION,
        STATUS AS SCREENING_STATUS,
        METRICS:roc_auc::FLOAT AS HELD_OUT_ROC_AUC,
        METRICS:average_precision::FLOAT AS HELD_OUT_PR_AUC,
        METRICS:brier_score::FLOAT AS HELD_OUT_BRIER
    FROM {database_name}.{control_schema}.PIPELINE_ATTEMPT
    WHERE RELEASE_ID = ?
    ORDER BY COMPLETED_AT DESC
    """,
    params=[release_id],
).to_pandas()
display(attempts_df)
display(pd.DataFrame(prod_candidate["gate_table"]))

## Section 2 — Loop 3: Deterministic Screening vs. Accountable Owner Approval

A candidate with `SCREENING_STATUS = 'accepted'` has passed deterministic metric and segment floors, but **it is not approved and does not serve traffic**:

1. Only `CRISK_DEMO_PROD_OWNER` has `INSERT` privileges on `CONTROL.PROMOTION_DECISION`.
2. `CONTROL.ACTIVATE_SERVING` runs with `EXECUTE AS OWNER` (owned by `CRISK_DEMO_PROD_OWNER`) and **fails closed** if `PROMOTION_DECISION` does not contain an explicit `'approve'` row for the exact candidate version.
3. Recording approval in `CONTROL.PROMOTION_DECISION` still leaves `CONTROL.SERVING_SELECTION` unchanged until Loop 4 activation runs.

In [ ]:
# 1. Attempting activation before owner approval fails closed
session.use_role(service_role)
unapproved_attempt = activate_serving(
    session,
    candidate_version=prod_candidate["model_version"],
    decision_id="DEC_MISSING",
    mode="BATCH",
    candidate_weight=100,
    project=project,
)
print("Unapproved activation check:", unapproved_attempt["status_message"])

# 2. Accountable production owner records explicit approval in CONTROL.PROMOTION_DECISION
session.use_role(prod_owner_role)
approval = record_promotion_decision(
    session,
    run_id=prod_candidate["run_id"],
    candidate_version=prod_candidate["model_version"],
    incumbent_version=None,
    screening_status=prod_candidate["screening_status"],
    exact_candidate_smoke_passed=True,
    rollback_ready=True,
    project=project,
)

decisions_df = session.sql(
    f"""
    SELECT
        DECISION_ID,
        RUN_ID,
        CANDIDATE_VERSION,
        INCUMBENT_VERSION,
        DECISION,
        CHECKS,
        DECIDED_BY,
        DECIDED_AT
    FROM {database_name}.{control_schema}.PROMOTION_DECISION
    WHERE DECISION_ID = ?
    """,
    params=[approval["decision_id"]],
).to_pandas()
display(decisions_df)

## Section 3 — Loop 4 (Recommended Default): Warehouse Batch Inference

For most tabular MLOps workloads, **warehouse batch inference** is the recommended default:

1. `CALL CONTROL.ACTIVATE_SERVING(version, decision_id, 'BATCH', 100)` verifies the approval row in `CONTROL.PROMOTION_DECISION`, moves the `PRODUCTION` alias (`ALTER MODEL ... VERSION ... SET ALIAS = PRODUCTION`), and writes the active selector to `CONTROL.SERVING_SELECTION`.
2. Scheduled batch scoring resolves `version("PRODUCTION")` from the Model Registry, retrieves point-in-time features from `PROD_FEATURE_STORE`, executes `mv.run(..., function_name="predict_proba")` on the virtual warehouse, and appends durable prediction rows (`PREDICTION_ID`, `ACCOUNT_ID`, `OBSERVATION_DATE`, `MODEL_VERSION`, `SCORE`) to `CONTROL.PREDICTION_RECORD`.

In [ ]:
session.use_role(service_role)
batch_activation = activate_serving(
    session,
    candidate_version=prod_candidate["model_version"],
    decision_id=approval["decision_id"],
    mode="BATCH",
    candidate_weight=100,
    project=project,
)
print("Batch activation:", batch_activation["status_message"])

serving_df = session.sql(
    f"""
    SELECT
        SELECTION_ID,
        ENVIRONMENT,
        MODEL_NAME,
        MODEL_VERSION,
        FEATURE_CONTRACT,
        DECISION_ID,
        VERIFICATION_STATUS,
        ACTIVE,
        SELECTED_AT
    FROM {database_name}.{control_schema}.SERVING_SELECTION
    WHERE ENVIRONMENT = 'PROD'
    ORDER BY SELECTED_AT DESC
    """
).to_pandas()
display(serving_df)

scoring_dates = [
    "2025-10-31",
    "2025-11-30",
    "2025-12-31",
    "2026-01-31",
    "2026-02-28",
    "2026-03-31",
    "2026-06-30",
    "2026-07-31",
]
batch_summaries = [
    run_warehouse_batch_scoring(
        session,
        observation_date=obs_date,
        release_id=release_id,
        environment_name="PROD",
        model_alias="PRODUCTION",
        project=project,
    )
    for obs_date in scoring_dates
]
display(pd.DataFrame(batch_summaries))

## Section 4 — Loop 4 (Online Reference Branch): Stable Gateway (`SHADOW` vs. `SPLIT`)

When synchronous REST latency is required, Model Registry aliases do not retarget an existing container service. Instead, treat the **Gateway (`CREATE GATEWAY`) as stable** and **SPCS model services as disposable**:

1. **Create & smoke-test the candidate service (`jobs/deploy_service.py`):** Executed inside Snowflake via `EXECUTE CODE BUNDLE ... ENTRYPOINT = 'jobs/deploy_service.py'` so CI never installs `snowflake-ml-python`. Creates the candidate service with `autocapture=True` and writes `'SMOKE_TESTED'` to `CONTROL.SERVICE_DEPLOYMENT`.
2. **Option A — Shadow Model (`mode='SHADOW'`, `type: shadow_traffic`, Preview):** The incumbent (`primary`) continues serving `100%` of caller responses while `weight%` of live requests is mirrored to the challenger (`shadow`). Auto Capture logs both endpoints so a **Gateway Model Monitor** (`GATEWAY = ...`) can compare drift and accuracy (`BASE_SERVICE`) with zero production impact.
3. **Option B — Traffic Split (`mode='SPLIT'`, `type: traffic_split`, GA):** Routes live traffic across endpoints with weights summing to `100` (e.g., `90/10` canary → `100%` cutover), keeping the incumbent service warm during the rollback window.

In [ ]:
# Train and approve a challenger PROD candidate (v2) to compare against incumbent (v1)
release_id_v2 = "R_2026_10_15"
session.use_role(service_role)
prod_candidate_v2 = train_and_register_candidate(
    session,
    environment_name="PROD",
    release_id=release_id_v2,
    dataset_version=dataset_version,
    source_state="main@2026-10-15",
    bundle_name=f"CRISK_DEMO_TRAIN_{release_id_v2}",
)

session.use_role(prod_owner_role)
approval_v2 = record_promotion_decision(
    session,
    run_id=prod_candidate_v2["run_id"],
    candidate_version=prod_candidate_v2["model_version"],
    incumbent_version=prod_candidate["model_version"],
    screening_status=prod_candidate_v2["screening_status"],
    exact_candidate_smoke_passed=True,
    rollback_ready=True,
    project=project,
)

# Verify that online SHADOW activation fails closed before jobs/deploy_service.py writes SMOKE_TESTED
session.use_role(service_role)
missing_service_check = activate_serving(
    session,
    candidate_version=prod_candidate_v2["model_version"],
    decision_id=approval_v2["decision_id"],
    mode="SHADOW",
    candidate_weight=10,
    project=project,
)
print("Online activation before smoke-test receipt:", missing_service_check["status_message"])

incumbent_svc = f"{database_name}.PROD.CREDIT_DEFAULT_SVC_{prod_candidate['model_version']}!inference"
challenger_svc = f"{database_name}.PROD.CREDIT_DEFAULT_SVC_{prod_candidate_v2['model_version']}!inference"

shadow_spec = f"""spec:
  type: shadow_traffic
  primary:
  - type: endpoint
    value: {incumbent_svc}
  shadow:
  - type: endpoint
    value: {challenger_svc}
    weight: 10"""

split_spec = f"""spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: {incumbent_svc}
    weight: 90
  - type: endpoint
    value: {challenger_svc}
    weight: 10"""

print("\n--- Option A: Gateway Shadow Model Specification (Preview) ---")
print(shadow_spec)
print("\n--- Option B: Gateway Traffic Split Specification (GA) ---")
print(split_spec)

## Section 5 — Stages 6 & 7: Model Monitoring, Delayed 90-Day Ground Truth & Rollback

Production monitoring separates **three distinct questions**:

1. **System health:** Did scheduled tasks, services, and monitors complete within budget?
2. **Data & prediction drift:** Have feature distributions or predicted scores shifted against baseline in **Model Version Monitors** (`SOURCE = ...`, GA) or **Gateway Model Monitors** (`GATEWAY = ...`, Preview)? Drift triggers investigation, never unconditional retraining.
3. **True outcomes (90-day delayed finality):** Join `CONTROL.PREDICTION_RECORD` to `RAW.ACCOUNT_OBSERVATION` (`CONTROL.V_PREDICTION_OUTCOME_MATURITY`). Cohorts whose 90-day window has not elapsed (`replay_date < OUTCOME_FINALITY_DATE`) are classified as **`PENDING`**—never treated as negative (`0`).

In [ ]:
replay_checkpoints = [
    ("2026-02-15", True),   # Drift visible in Jan 2026 scores; Jan outcomes still PENDING
    ("2026-05-15", True),   # Jan-Feb 2026 drift cohorts have reached 90-day FINAL maturity
    ("2026-09-01", True),   # Full simulated horizon: through-May FINAL; June-July PENDING
]

replay_rows = []
for r_date, drift_flag in replay_checkpoints:
    snap = evaluate_delayed_outcomes(
        session,
        replay_date=r_date,
        model_version=prod_candidate["model_version"],
        drift_detected=drift_flag,
        project=project,
    )
    replay_rows.append(
        {
            "REPLAY_DATE": snap["replay_date"],
            "TOTAL_PREDICTIONS": snap["coverage"]["predictions"],
            "MATCHED_FINAL": snap["coverage"]["matched_final"],
            "PENDING_90D_WINDOW": snap["coverage"]["pending"],
            "UNMATCHED": snap["coverage"]["unmatched"],
            "MATURED_ROC_AUC": snap["matured_metrics"]["roc_auc"],
            "MATURED_BRIER": snap["matured_metrics"]["brier_score"],
            "PERFORMANCE_DEGRADED": snap["performance_degraded"],
            "RECOMMENDED_ACTION": snap["recommendation"]["action"],
        }
    )

display(pd.DataFrame(replay_rows))

cohort_df = session.sql(
    f"""
    SELECT
        OBSERVATION_DATE,
        LABEL_MATURITY_STATUS,
        COUNT(*) AS PREDICTION_COUNT,
        AVG(SCORE)::FLOAT AS AVG_PREDICTED_SCORE,
        AVG(MATURED_DEFAULT_WITHIN_90D)::FLOAT AS MATURED_DEFAULT_RATE
    FROM {database_name}.{control_schema}.V_PREDICTION_OUTCOME_MATURITY
    WHERE MODEL_VERSION = ?
    GROUP BY OBSERVATION_DATE, LABEL_MATURITY_STATUS
    ORDER BY OBSERVATION_DATE, LABEL_MATURITY_STATUS
    """,
    params=[prod_candidate["model_version"]],
).to_pandas()
display(cohort_df)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pivot_counts = cohort_df.pivot(
    index="OBSERVATION_DATE",
    columns="LABEL_MATURITY_STATUS",
    values="PREDICTION_COUNT",
).fillna(0)
pivot_counts.plot(kind="bar", stacked=True, ax=axes[0], color=["#16a37a", "#29b5e8", "#94a3b8"])
axes[0].set_title("Cohort Label Finality (Pending Labels Never Treated as 0)")
axes[0].set_xlabel("Observation Date")
axes[0].set_ylabel("Scored Accounts")
axes[0].tick_params(axis="x", rotation=45)

monthly_rates = (
    cohort_df.groupby("OBSERVATION_DATE", as_index=False)
    .agg(
        AVG_PREDICTED_SCORE=("AVG_PREDICTED_SCORE", "mean"),
        MATURED_DEFAULT_RATE=("MATURED_DEFAULT_RATE", "mean"),
    )
    .sort_values("OBSERVATION_DATE")
)
axes[1].plot(
    monthly_rates["OBSERVATION_DATE"].astype(str),
    monthly_rates["AVG_PREDICTED_SCORE"],
    marker="o",
    label="Mean Predicted Score",
    color="#003566",
)
axes[1].plot(
    monthly_rates["OBSERVATION_DATE"].astype(str),
    monthly_rates["MATURED_DEFAULT_RATE"],
    marker="s",
    label="Matured 90d Default Rate (Final Only)",
    color="#e63946",
)
axes[1].axvline("2026-01-31", color="#64748b", linestyle="--", label="Drift Regime Start")
axes[1].set_title("Predicted Score vs. Matured 90-Day Default Rate")
axes[1].set_xlabel("Observation Date")
axes[1].set_ylabel("Rate / Probability")
axes[1].tick_params(axis="x", rotation=45)
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# 1. Record governed operational decision in CONTROL.OPERATIONAL_ACTION
session.use_role(prod_owner_role)
action_record = record_operational_action(
    session,
    model_version=prod_candidate["model_version"],
    action=replay_rows[-1]["RECOMMENDED_ACTION"],
    rationale="Post-2026-01 drift regime confirmed once 90-day outcomes matured; schedule candidate review.",
    project=project,
)
print("Recorded operational action:", action_record)

# 2. Activate v2 in BATCH mode, then execute full-path rollback via RECOVER_SERVING
session.use_role(service_role)
v2_activation = activate_serving(
    session,
    candidate_version=prod_candidate_v2["model_version"],
    decision_id=approval_v2["decision_id"],
    mode="BATCH",
    candidate_weight=100,
    project=project,
)
print("Activated v2:", v2_activation["status_message"])

recovery = recover_serving(
    session,
    environment_name="PROD",
    project=project,
)
print("Rollback result:", recovery["status_message"])

history_df = session.sql(
    f"""
    SELECT
        SELECTION_ID,
        MODEL_VERSION,
        FEATURE_CONTRACT,
        PREVIOUS_SELECTION_ID,
        VERIFICATION_STATUS,
        RECOVERY_STATUS,
        ACTIVE,
        SELECTED_AT
    FROM {database_name}.{control_schema}.SERVING_SELECTION
    WHERE ENVIRONMENT = 'PROD'
    ORDER BY SELECTED_AT DESC
    """
).to_pandas()
display(history_df)